In [5]:
from chefai.parser import *
import rich

In [6]:


# ---------------------------------------------------
#  Integrazione nella funzione principale (solo snippet)
# ---------------------------------------------------
def extract_recipes_from_pdf(pdf_path: str | Path, output_json: str = "ricette.json") -> None:
    # ... (stesso codice di prima per estrarre page_texts con fitz)

    page_texts = {}
    doc = fitz.open(pdf_path)
    for i in tqdm(range(len(doc)), desc="Estrazione testo"):
        page_texts[i+1] = doc[i].get_text("text")
    doc.close()
    return page_texts

In [7]:
text = extract_recipes_from_pdf("./data/La-cucina-italiana.pdf")


Estrazione testo: 100%|██████████| 912/912 [00:01<00:00, 556.77it/s]


In [8]:
text[2] # author and title
rich.print(text[5]) # table of contents

SOMMARIO
Prefazione
La cucina italiana
Cucina di mare e cucina di terra
Gualtiero Marchesi – biografia
Ricette
Preparazioni di base
Antipasti di mare
Antipasti dell’entroterra
Primi di mare
Primi dell’entroterra
Pizze e focacce
Secondi di mare
Secondi dell’entroterra
Pesci d’acqua dolce, lumache e rane
Verdure
Dolci
Indici
Indice alfabetico delle ricette

In [12]:
import re
from pathlib import Path
from typing import Optional

# Aggiungi questa funzione di utilità
def find_toc_text(pdf_path: str | Path) -> Optional[str]:
    """
    Cerca la pagina/indice che contiene l'elenco dei capitoli principali
    e restituisce il blocco di testo più probabile.
    """
    import fitz  # PyMuPDF
    
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF non trovato: {pdf_path}")
    
    doc = fitz.open(pdf_path)
    
    target_chapters = {
        "Preparazioni di base",
        "Antipasti di mare",
        "Antipasti dell’entroterra",
        "Primi di mare",
        "Primi dell’entroterra",
        "Pizze e focacce",
        "Secondi di mare",
        "Secondi dell’entroterra",
        "Pesci d’acqua dolce, lumache e rane",
        "Verdure",
        "Dolci"
    }
    
    best_match = None
    best_score = 0
    best_page = None
    best_text = ""
    
    # Di solito l'indice è nelle prime 5–20 pagine
    for page_num in range(min(40, len(doc))):
        page = doc[page_num]
        text = page.get_text("text")
        
        # Normalizziamo un po' (ma non troppo)
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        
        found = set()
        for line in lines:
            clean = re.sub(r'\s+', ' ', line).strip()
            if clean in target_chapters:
                found.add(clean)
        
        score = len(found)
        
        if score > best_score:
            best_score = score
            best_page = page_num + 1
            best_text = text
            best_match = found
            
            # Se troviamo quasi tutto → possiamo fermarci presto
            if score >= len(target_chapters) - 2:
                break
    
    doc.close()
    
    if best_score >= 6:  # soglia ragionevole per un indice reale
        print(f"Trovato probabile indice a pagina {best_page} (score: {best_score}/{len(target_chapters)} capitoli)")
        print(f"Capitoli trovati: {sorted(best_match)}")
        return best_text.strip()
    else:
        print("Indice non trovato con soglia sufficiente.")
        print(f"Miglior candidato (score {best_score}):")
        print(best_text[:800] + "..." if len(best_text) > 800 else best_text)
        return None


# Uso
toc_content = find_toc_text("./data/La-cucina-italiana.pdf")
if toc_content:
    # Qui puoi splittare ulteriormente o salvarlo
    Path("indice_trovato.md").write_text(toc_content, encoding="utf-8")

Trovato probabile indice a pagina 5 (score: 11/11 capitoli)
Capitoli trovati: ['Antipasti dell’entroterra', 'Antipasti di mare', 'Dolci', 'Pesci d’acqua dolce, lumache e rane', 'Pizze e focacce', 'Preparazioni di base', 'Primi dell’entroterra', 'Primi di mare', 'Secondi dell’entroterra', 'Secondi di mare', 'Verdure']


In [ ]:
chapter_titles = [
    "PREPARAZIONI DI BASE",
    "ANTIPASTI DI MARE",
    "ANTIPASTI DELL’ENTROTERRA",
    "PRIMI DI MARE",
    "PRIMI DELL’ENTROTERRA",
    "PIZZE E FOCACCE",
    "SECONDI DI MARE",
    "SECONDI DELL’ENTROTERRA",
    "PESCI D’ACQUA DOLCE, LUMACHE E RANE",
    "VERDURE",
    "DOLCI"
]

def normalize(s: str) -> str:
    """Normalizza per ignorare spazi multipli, maiuscole/minuscole, punteggiatura finale"""
    return ' '.join(s.strip().upper().split())

normalized_chapters = [normalize(t) for t in chapter_titles]

chapter_positions = {}


for pagina, contenuto in sorted(text.items()):
    normalized_line = ' '.join(contenuto.strip().upper().split())
    if normalized_line in normalized_chapters:
        print(f"Capitolo {normalized_line} inizia alla pagina {pagina}")

# Stampa risultati
for titolo, posizioni in sorted(chapter_positions.items(), key=lambda x: x[1]):
    print(f"{titolo:40} → posizioni: {posizione}")

Capitolo PREPARAZIONI DI BASE inizia alla pagina 15
Capitolo ANTIPASTI DI MARE inizia alla pagina 23
Capitolo ANTIPASTI DELL’ENTROTERRA inizia alla pagina 31
Capitolo PRIMI DI MARE inizia alla pagina 68
Capitolo PRIMI DELL’ENTROTERRA inizia alla pagina 94
Capitolo PIZZE E FOCACCE inizia alla pagina 284
Capitolo SECONDI DI MARE inizia alla pagina 310
Capitolo SECONDI DELL’ENTROTERRA inizia alla pagina 413
Capitolo PESCI D’ACQUA DOLCE, LUMACHE E RANE inizia alla pagina 614
Capitolo VERDURE inizia alla pagina 628
Capitolo DOLCI inizia alla pagina 765


In [30]:
capitoli_inizio = {
    "PREPARAZIONI DI BASE":                  15,
    "ANTIPASTI DI MARE":                     23,
    "ANTIPASTI DELL’ENTROTERRA":             31,
    "PRIMI DI MARE":                         68,
    "PRIMI DELL’ENTROTERRA":                 94,
    "PIZZE E FOCACCE":                      284,
    "SECONDI DI MARE":                      310,
    "SECONDI DELL’ENTROTERRA":              413,
    "PESCI D’ACQUA DOLCE, LUMACHE E RANE":  614,
    "VERDURE":                              628,
    "DOLCI":                                765,
}

# Aggiungiamo la fine del libro come limite
fine_libro = 856

# Creiamo una lista ordinata di (titolo, pagina_inizio)
inizio_ordinati = sorted(capitoli_inizio.items(), key=lambda x: x[1])

# Costruiamo gli intervalli
capitoli_intervalli = []

for i in range(len(inizio_ordinati)):
    titolo, pagina_start = inizio_ordinati[i]
    
    if i + 1 < len(inizio_ordinati):
        pagina_end = inizio_ordinati[i + 1][1] - 1
    else:
        pagina_end = fine_libro
    
    capitoli_intervalli.append({
        "titolo": titolo,
        "pagina_inizio": pagina_start,
        "pagina_fine": pagina_end,
        "numero_pagine": pagina_end - pagina_start + 1
    })

In [31]:
capitoli_testo = {}

for cap in capitoli_intervalli:
    titolo = cap["titolo"]
    start = cap["pagina_inizio"]
    end   = cap["pagina_fine"]
    
    testo_completo = []
    
    for p in range(start, end + 1):
        if p in text:
            testo_pagina = text[p].strip()
            if testo_pagina:  # saltiamo pagine vuote o quasi vuote
                testo_completo.append(testo_pagina)
    
    testo_unito = "\n\n".join(testo_completo)
    
    # Opzionale: rimuovi eventuali marker di pagina se li hai aggiunti prima
    testo_unito = testo_unito.replace("\n\n--- PAGINA ", "\n\n")
    
    capitoli_testo[titolo] = testo_unito
    
    print(f"{titolo:36}  →  {start:4}–{end:4}  ({len(testo_completo)} pagine)")

PREPARAZIONI DI BASE                  →    15–  22  (8 pagine)
ANTIPASTI DI MARE                     →    23–  30  (8 pagine)
ANTIPASTI DELL’ENTROTERRA             →    31–  67  (37 pagine)
PRIMI DI MARE                         →    68–  93  (26 pagine)
PRIMI DELL’ENTROTERRA                 →    94– 283  (190 pagine)
PIZZE E FOCACCE                       →   284– 309  (26 pagine)
SECONDI DI MARE                       →   310– 412  (103 pagine)
SECONDI DELL’ENTROTERRA               →   413– 613  (201 pagine)
PESCI D’ACQUA DOLCE, LUMACHE E RANE   →   614– 627  (14 pagine)
VERDURE                               →   628– 764  (137 pagine)
DOLCI                                 →   765– 856  (92 pagine)


In [33]:
from pathlib import Path
import re

# Mappa capitoli (dal tuo output precedente)
capitoli_inizio = {
    "PREPARAZIONI DI BASE":                  15,
    "ANTIPASTI DI MARE":                     23,
    "ANTIPASTI DELL’ENTROTERRA":             31,
    "PRIMI DI MARE":                         68,
    "PRIMI DELL’ENTROTERRA":                 94,
    "PIZZE E FOCACCE":                      284,
    "SECONDI DI MARE":                      310,
    "SECONDI DELL’ENTROTERRA":              413,
    "PESCI D’ACQUA DOLCE, LUMACHE E RANE":  614,
    "VERDURE":                              628,
    "DOLCI":                                765,
}

FINE_LIBRO = 856

# Cartella di output
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ordina i capitoli per pagina di inizio
inizio_ordinati = sorted(capitoli_inizio.items(), key=lambda x: x[1])

capitoli_salvati = []

for i, (titolo, pagina_start) in enumerate(inizio_ordinati):
    # Calcola fine del capitolo
    if i + 1 < len(inizio_ordinati):
        pagina_end = inizio_ordinati[i + 1][1] - 1
    else:
        pagina_end = FINE_LIBRO
    
    # Raccogli tutte le pagine del capitolo
    pagine_testo = []
    for p in range(pagina_start, pagina_end + 1):
        if p in text:
            testo_p = text[p].rstrip()
            if testo_p.strip():
                pagine_testo.append(f"--- pagina {p} ---\n{testo_p}")
    
    if not pagine_testo:
        print(f"Nessun contenuto trovato per {titolo} (pagine {pagina_start}–{pagina_end})")
        continue
    
    contenuto = "\n\n".join(pagine_testo)
    
    # Nome file sicuro
    nome_base = re.sub(r'[^a-z0-9]+', '-', titolo.lower())
    nome_file = f"{nome_base}.md"
    percorso = OUTPUT_DIR / nome_file
    
    # Contenuto Markdown
    md_content = []
    md_content.append("---")
    md_content.append(f"titolo: {titolo}")
    md_content.append(f"pagine: {pagina_start}–{pagina_end}")
    md_content.append(f"numero_pagine: {pagina_end - pagina_start + 1}")
    md_content.append("---")
    md_content.append("")
    md_content.append(f"# {titolo}")
    md_content.append("")
    md_content.append(f"**Pagine:** {pagina_start} – {pagina_end}")
    md_content.append("")
    md_content.append("```text")
    md_content.append(contenuto)
    md_content.append("```")
    
    # Salva
    percorso.write_text("\n".join(md_content), encoding="utf-8")
    
    print(f"Salvato: {percorso}  ({pagina_start}–{pagina_end}, {len(pagine_testo)} pagine)")
    capitoli_salvati.append(titolo)

print(f"\nCompletato: {len(capitoli_salvati)} capitoli salvati in {OUTPUT_DIR}")

Salvato: data\processed\preparazioni-di-base.md  (15–22, 8 pagine)
Salvato: data\processed\antipasti-di-mare.md  (23–30, 8 pagine)
Salvato: data\processed\antipasti-dell-entroterra.md  (31–67, 37 pagine)
Salvato: data\processed\primi-di-mare.md  (68–93, 26 pagine)
Salvato: data\processed\primi-dell-entroterra.md  (94–283, 190 pagine)
Salvato: data\processed\pizze-e-focacce.md  (284–309, 26 pagine)
Salvato: data\processed\secondi-di-mare.md  (310–412, 103 pagine)
Salvato: data\processed\secondi-dell-entroterra.md  (413–613, 201 pagine)
Salvato: data\processed\pesci-d-acqua-dolce-lumache-e-rane.md  (614–627, 14 pagine)
Salvato: data\processed\verdure.md  (628–764, 137 pagine)
Salvato: data\processed\dolci.md  (765–856, 92 pagine)

Completato: 11 capitoli salvati in data\processed


In [37]:
capitoli_salvati

['PREPARAZIONI DI BASE',
 'ANTIPASTI DI MARE',
 'ANTIPASTI DELL’ENTROTERRA',
 'PRIMI DI MARE',
 'PRIMI DELL’ENTROTERRA',
 'PIZZE E FOCACCE',
 'SECONDI DI MARE',
 'SECONDI DELL’ENTROTERRA',
 'PESCI D’ACQUA DOLCE, LUMACHE E RANE',
 'VERDURE',
 'DOLCI']

In [36]:
from chefai.extractor import *

In [41]:
# read md file in 'data/processed/preparazioni-di-base.md'
testo = Path("data/processed/preparazioni-di-base.md").read_text(encoding="utf-8")
# rich.print(testo)